In [ ]:
import contextlib
import json
import math
import os
import subprocess
import sys
from pathlib import Path

# Hindi Indic ASR pipeline

This notebook runs `pipeline.yaml` directly through Curator's shared Hydra runner, following the ALM and Qwen ASR tutorial method. The YAML contains the complete Indic Canary primary-ASR, Parakeet TensorRT recovery, selection, cleanup, and WER graph used by both benchmark entries. No tutorial-specific Python wrapper is involved.

## Step 1: Install and provision the runtimes

From the repository root:

```bash
uv sync --frozen --extra audio_canary_trtllm --no-default-groups
source .venv/bin/activate
python -m nemo_curator.stages.audio.inference.scripts.install_indic_canary_trtllm_runtime
```

Curator and Parakeet run in the parent environment. Indic Canary launches TensorRT-LLM from the separately locked CPython 3.12/CUDA 13 child runtime.

## Step 2: Prepare the pinned public Hindi cohort

```bash
python benchmarking/data_prep/prepare_audio_indic_asr_data.py \
  --output-path /data/audio_indic_asr_parakeet_hindi_531h_35376a11 \
  --cache-dir /data/_hf_cache/audio_indic_asr \
  --subset-hours 1 \
  --subset-manifest /data/audio_indic_asr_parakeet_hindi_531h_35376a11/manifest-1h.jsonl
```

The setup pins `ketav/parakeet-hindi-asr` revision `35376a112c4b79318eeaba0c0dd1b6f1a9bf0ea0` and validates 216,169 clips / 531.7738 audio hours. It also writes a deterministic at-least-one-hour prefix of that same cohort for local functional validation.

In [ ]:
REPO_ROOT = Path.cwd().resolve()
DATASET_DIR = Path("/data/audio_indic_asr_parakeet_hindi_531h_35376a11")
MANIFEST_PATH = DATASET_DIR / "manifest.jsonl"  # use manifest-1h.jsonl for a local functional run
CANARY_ENGINE = Path("/models/audio_indic_asr/indic_canary_trtllm/engine_bfloat16_64_new")
PARAKEET_ENGINE = Path("/models/audio_indic_asr/parakeet_indic_trt/encoder_fp16_b1_o8_m16_f8_o800_m4001")
OUTPUT_ROOT = REPO_ROOT / "indic_asr_tutorial_output"

required = [MANIFEST_PATH, CANARY_ENGINE, PARAKEET_ENGINE]
missing = [str(path) for path in required if not path.exists()]
if missing:
    message = f"Update the tutorial paths; missing: {missing}"
    raise FileNotFoundError(message)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## Step 3: Define the direct YAML runner

Both invocations use the same data, engines, stage parameters, worker counts, and output schema. Only `backend` and the fresh output path change. The canonical benchmark harness separately owns sharding, GPU recording, wall-clock measurement, and acceptance gates.

In [ ]:
def run_backend(backend: str, run_number: int = 1) -> Path:
    output_path = OUTPUT_ROOT / f"{backend}-{run_number:03d}.jsonl"
    if output_path.exists():
        message = f"Choose a fresh output path: {output_path}"
        raise FileExistsError(message)
    command = [
        sys.executable,
        str(REPO_ROOT / "nemo_curator/config/run.py"),
        "--config-path",
        "../../tutorials/audio/indic_asr",
        "--config-name",
        "pipeline",
        f"dataset_dir={DATASET_DIR}",
        f"manifest_path={MANIFEST_PATH}",
        f"indic_canary_engine_dir={CANARY_ENGINE}",
        f"parakeet_tensorrt_engine_dir={PARAKEET_ENGINE}",
        f"output_path={output_path}",
        f"backend={backend}",
        "execution_mode=streaming",
    ]
    environment = os.environ | {
        "RAY_DATA_OP_RESERVATION_RATIO": "0",
        "RAY_DATA_STREAMING_MAX_BUFFER_SIZE_MB": "4096",
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    }
    subprocess.run(command, cwd=REPO_ROOT, env=environment, check=True)  # noqa: S603
    return output_path

## Step 4: Run Xenna

The benchmark's 600–900 second EOS criterion applies to harness wall time on the standard eight-H100 runner, not to notebook timing.

In [ ]:
xenna_output = run_backend("xenna")
xenna_output

## Step 5: Run Ray Data

Wait for the Xenna process and its Ray cluster to stop, then run the same YAML with Ray Data.

In [ ]:
ray_data_output = run_backend("ray_data")
ray_data_output

## Step 6: Compare functional outputs

The summary below checks identity, duration, prediction, provenance, and WER coverage without loading the full manifests into memory.

In [ ]:
def summarize(path: Path) -> dict:
    identities = {}
    coverage = dict.fromkeys(
        (
            "primary_model_prediction",
            "fallback_model_prediction",
            "best_prediction",
            "cleaned_text",
            "abbreviated_text",
        ),
        0,
    )
    recognized_sources = 0
    finite_wer = 0
    with path.open(encoding="utf-8") as stream:
        for line in stream:
            if not line.strip():
                continue
            row = json.loads(line)
            identity = row["audio_item_id"]
            if identity in identities:
                message = f"Duplicate output identity: {identity}"
                raise RuntimeError(message)
            identities[identity] = (row["audio_filepath"], float(row["duration"]))
            for key in coverage:
                coverage[key] += isinstance(row.get(key), str) and bool(row[key].strip())
            recognized_sources += row.get("best_prediction_source") in {"primary", "fallback"}
            with contextlib.suppress(KeyError, TypeError, ValueError):
                finite_wer += math.isfinite(float(row["wer_pct"]))
    count = len(identities)
    return {
        "rows": count,
        "audio_hours": math.fsum(value[1] for value in identities.values()) / 3600,
        **{f"{key}_coverage": value / count for key, value in coverage.items()},
        "recognized_source_coverage": recognized_sources / count,
        "wer_coverage": finite_wer / count,
        "_identities": identities,
    }


xenna_summary = summarize(xenna_output)
ray_data_summary = summarize(ray_data_output)
xenna_identities = xenna_summary.pop("_identities")
ray_data_identities = ray_data_summary.pop("_identities")
if xenna_identities != ray_data_identities:
    message = "Xenna and Ray Data outputs do not contain identical inputs"
    raise RuntimeError(message)
{"xenna": xenna_summary, "ray_data": ray_data_summary}